02_summarization_gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/02_summarization_gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/02_summarization_gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Summarization with Gemini: Condensing Business Documents

## What you will learn
- Use an LLM to summarize documents for business audiences.
- Compare LLM summaries against human-written reference summaries.
- Identify summarization failures: omission, hallucination, misleading emphasis.

**NLP Task**: Summarization — condensing longer content into shorter, actionable form.

**Dataset**: `EdinburghNLP/xsum` from HuggingFace — BBC news articles with
single-sentence reference summaries. We'll treat these as "executive briefs."

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [3]:
%pip install -q google-genai datasets pandas openpyxl


In [4]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


Using Gemini model: gemini-3.1-flash-lite-preview
Gemini client ready.


## Load Dataset

XSum contains BBC articles with one-sentence reference summaries.
We take 10 articles to keep demo fast.


In [5]:
from datasets import load_dataset

ds = load_dataset('EdinburghNLP/xsum', split='test')

# Take 10 varied articles (skip very short ones)
import random
random.seed(42)

candidates = [i for i in range(len(ds)) if len(ds[i]['document']) > 500]
sample_indices = random.sample(candidates, 10)

sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df = sample_df.rename(columns={'document': 'article', 'summary': 'reference_summary'})

print(f'Sample size: {len(sample_df)}')
for i, row in sample_df.head(3).iterrows():
    print(f'\nArticle {i} ({len(row["article"])} chars):')
    print(f'  First 150 chars: {row["article"][:150]}...')
    print(f'  Reference summary: {row["reference_summary"]}')


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

Sample size: 10

Article 0 (1648 chars):
  First 150 chars: The cinema chain has "provided the British public with unforgettable, cinema experiences", Bafta said.
The awards will be held at London's Royal Alber...
  Reference summary: Curzon will receive an outstanding British contribution to cinema prize at this year's Bafta film awards.

Article 1 (2980 chars):
  First 150 chars: James Warnock, 56, has been convicted of the "horrifying" killing of Yiannoulla Yianni, 17, in 1982.
She was attacked while home alone in Hampstead, n...
  Reference summary: A man who described himself in court as looking like John Travolta has been found guilty of the rape and murder of a teenager 34 years ago.

Article 2 (1265 chars):
  First 150 chars: Morgan Geyser and Anissa Weier have been charged with attempted murder over the attack.
Last week, a Wisconsin judge decided that they should be tried...
  Reference summary: Two 13-year-old girls accused of stabbing a classmate to please the online horror

## Summarize with Gemini

We ask Gemini for a 1-2 sentence executive summary, similar to
"Summarize this report in one sentence for the board."


In [6]:
SUMMARIZATION_PROMPT = """You are writing an executive brief. Summarize the following article
in exactly 1-2 sentences. Focus on the most important finding or event.
Do not add any information not present in the article.

Article:
\"\"\"{article}\"\"\"

Summary:"""

def summarize_text(article: str) -> dict:
    prompt = SUMMARIZATION_PROMPT.format(article=article)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return {'summary': resp.text.strip(), 'error': None}
    except Exception as e:
        return {'summary': None, 'error': str(e)}

results = []
for idx, row in sample_df.iterrows():
    result = summarize_text(row['article'])
    results.append(result)
    time.sleep(0.5)

sample_df['llm_summary'] = [r['summary'] for r in results]
sample_df['error'] = [r['error'] for r in results]

print('Summarization complete.')


Summarization complete.


## Evaluate Summaries with Code-Based Checks

We use simple, deterministic checks that anyone can verify.
These are useful because they catch obvious failures without needing human review.


In [7]:
# Check 1: Length — is it actually concise?
sample_df['llm_word_count'] = sample_df['llm_summary'].fillna('').apply(lambda x: len(x.split()))
sample_df['ref_word_count'] = sample_df['reference_summary'].apply(lambda x: len(x.split()))
sample_df['length_ok'] = sample_df['llm_word_count'] <= 60  # 1-2 sentences should be < 60 words

# Check 2: Sentence count — did it follow the 1-2 sentence instruction?
sample_df['sentence_count'] = sample_df['llm_summary'].fillna('').apply(
    lambda x: len([s for s in x.split('.') if s.strip()])
)
sample_df['sentence_count_ok'] = sample_df['sentence_count'] <= 3

# Check 3: Overlap — does the summary share key words with the article?
def word_overlap(article, summary):
    if not summary:
        return 0.0
    article_words = set(article.lower().split())
    summary_words = set(summary.lower().split())
    # Remove common stop words for a rough content overlap
    stop = {'the','a','an','is','are','was','were','in','on','at','to','for','of','and','or','but','with','that','this','it'}
    article_content = article_words - stop
    summary_content = summary_words - stop
    if not summary_content:
        return 0.0
    return len(summary_content & article_content) / len(summary_content)

sample_df['content_overlap'] = sample_df.apply(
    lambda r: word_overlap(r['article'], r['llm_summary']), axis=1
)

print('=== Code-Based Checks ===')
print(f'Length OK (≤60 words):       {sample_df["length_ok"].mean():.0%}')
print(f'Sentence count OK (≤3):      {sample_df["sentence_count_ok"].mean():.0%}')
print(f'Avg LLM summary length:      {sample_df["llm_word_count"].mean():.0f} words')
print(f'Avg reference length:         {sample_df["ref_word_count"].mean():.0f} words')
print(f'Avg content word overlap:     {sample_df["content_overlap"].mean():.0%}')
print()
print('Low overlap (<50%) may indicate the LLM introduced new terminology or missed key topics.')


=== Code-Based Checks ===
Length OK (≤60 words):       100%
Sentence count OK (≤3):      90%
Avg LLM summary length:      46 words
Avg reference length:         20 words
Avg content word overlap:     53%

Low overlap (<50%) may indicate the LLM introduced new terminology or missed key topics.


## Side-by-Side Comparison

Read through these carefully. For each article, consider:
- Does the LLM summary capture the **main point**?
- Does it **add anything** that wasn't in the article? (hallucination)
- Does it **leave out** something critical? (omission)
- Does it give a **misleading impression** of what the article says?


In [8]:
for idx, row in sample_df.iterrows():
    print(f'--- Article {idx} ({row["llm_word_count"]} words, overlap: {row["content_overlap"]:.0%}) ---')
    print(f'Reference:  {row["reference_summary"]}')
    print(f'LLM:        {row["llm_summary"]}')
    print()


--- Article 0 (45 words, overlap: 68%) ---
Reference:  Curzon will receive an outstanding British contribution to cinema prize at this year's Bafta film awards.
LLM:        Bafta has announced that the cinema chain Curzon will receive the Outstanding British Contribution to Cinema award for its eight decades of supporting independent and foreign-language film. The honor will be presented at the awards ceremony held at London's Royal Albert Hall on 12 February.

--- Article 1 (47 words, overlap: 51%) ---
Reference:  A man who described himself in court as looking like John Travolta has been found guilty of the rape and murder of a teenager 34 years ago.
LLM:        James Warnock has been convicted of the 1982 murder of 17-year-old Yiannoulla Yianni after a DNA sample collected during a separate investigation matched evidence found at the original crime scene. The conviction brings closure to a long-unsolved cold case that had remained open for over four decades.

--- Article 2 (47 words

## Discussion Questions

1. **Was key information lost?** Compare the reference and LLM summaries.
   Where does the LLM lose important nuance?

2. **Was anything hallucinated?** Did the LLM add facts not in the article?
   This is the #1 summarization failure mode.

3. **Are the code-based checks enough?** Length and overlap catch some problems,
   but can a summary be the right length, use the right words, and still be wrong?
   What would you need to check that code alone can't?

## Export for Your Annotation

Download this Excel file and annotate each summary yourself.
For each row, fill in:
- **faithful** (YES/NO): Does it only contain info from the article?
- **captures_key_point** (YES/NO): Does it get the main idea right?
- **misleading** (YES/NO): Could it give a reader the wrong impression?
- **student_notes**: Your observations on what went right or wrong.


In [9]:
export_df = sample_df[['article', 'reference_summary', 'llm_summary', 'llm_word_count', 'content_overlap']].copy()
# Truncate article for readability in Excel
export_df['article_excerpt'] = export_df['article'].str[:500] + '...'
export_df = export_df.drop(columns=['article'])

# Add annotation columns for students
export_df['faithful'] = ''
export_df['captures_key_point'] = ''
export_df['misleading'] = ''
export_df['student_notes'] = ''

export_path = 'summarization_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')


Saved summarization_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>